## Meshes

In [ ]:
import os
import pickle
import numpy as np
import math
from pathlib import Path
from typing import List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import hypernetx as hnx
import sys

# Paths
sys.path.append("/home/infres/gailhard/SODA_Hypergraph-Generation-Features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/SODA_Hypergraph-Generation-Features")

from metrics import metrics_hypergraph as metrics
from metrics import metrics_mesh as metrics_mesh

# ---------------- CONFIG ----------------

DATASET_PATH = "SODA_Hypergraph-Generation-Features/data/manifoldNetAirplane.pkl"
OUTPUT_DIR = "featured_flow_matching_self_cond"
N_RUNS = 5
EPOCHS = 50
BATCH_SIZE = 32 # Reduced batch size for larger model
LR = 1e-4
SAMPLING_STEPS = 25
FEATURE_DIM = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- UTILS ----------------

def shuffle_hypergraph_indices(incidence, node_features):
    N, E = incidence.shape
    node_perm = np.random.permutation(N)
    edge_perm = np.random.permutation(E)
    return incidence[node_perm][:, edge_perm], node_features[node_perm]

def hypergraph_to_tensor(H: hnx.Hypergraph, F_dim=FEATURE_DIM):
    incidence = H.incidence_matrix().toarray().astype(np.float32)
    node_features = np.stack([H.nodes[n].feature[:F_dim] for n in H.nodes], axis=0).astype(np.float32)
    incidence, node_features = shuffle_hypergraph_indices(incidence, node_features)
    N, E = incidence.shape
    tensor = np.zeros((1, N, E + F_dim), dtype=np.float32)
    tensor[0, :, :E], tensor[0, :, E:] = incidence, node_features
    return tensor

def tensor_to_hypergraph(tensor, F_dim=FEATURE_DIM, threshold=0.5):
    if tensor.ndim == 4: tensor = tensor[0]
    _, N, E_total = tensor.shape
    E = E_total - F_dim
    # Convert from [-1, 1] Tanh range to [0, 1] then threshold
    incidence_raw = (tensor[0, :, :E] + 1) / 2
    incidence = (incidence_raw > threshold).astype(int)
    
    node_features = tensor[0, :, E:] 
    hyperedges = {e: np.where(incidence[:, e] == 1)[0].tolist() for e in range(E) if np.any(incidence[:, e] == 1)}
    
    H = hnx.Hypergraph(hyperedges)
    for i in range(N): H.add_node(i, feature=node_features[i])
    if len(H.edges) == 0: H.add_edge(0, [0])
    return H

def pad_tensor(tensor, max_N, max_E):
    C, N, E = tensor.shape
    padded = np.zeros((C, max_N, max_E), dtype=np.float32)
    padded[:, :N, :E] = tensor
    return padded

class HypergraphTensorDataset(Dataset):
    def __init__(self, hypergraphs: list, F_dim=FEATURE_DIM):
        self.max_N = max(len(H.nodes) for H in hypergraphs)
        self.max_E = max(len(H.edges) for H in hypergraphs) + F_dim
        self.tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H, F_dim), self.max_N, self.max_E)) 
                        for H in hypergraphs for _ in range(10)]
    def __len__(self): return len(self.tensors)
    def __getitem__(self, idx): return self.tensors[idx]

# ---------------- SCALED MODEL (6M+ PARAMS) ----------------

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
        )
    def forward(self, x):
        return x + self.net(x)

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device) * -math.log(10000) / (half - 1))
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([args.sin(), args.cos()], dim=-1)
        return self.mlp(emb)

class FlowMatchingModel(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # channels * 2 because of self-conditioning input
        self.input_conv = nn.Conv2d(channels * 2, 128, 3, padding=1)
        self.time_mlp = TimeEmbedding(256)
        self.t_proj = nn.Linear(256, 128)

        # Bottleneck architecture to hit ~6M params
        self.layers = nn.ModuleList([
            ResBlock(128), ResBlock(128),
            nn.Conv2d(128, 256, 3, padding=1),
            ResBlock(256), ResBlock(256), ResBlock(256),
            nn.Conv2d(256, 128, 3, padding=1),
            ResBlock(128), ResBlock(128)
        ])
        
        self.final_conv = nn.Sequential(
            nn.GroupNorm(8, 128),
            nn.SiLU(),
            nn.Conv2d(128, channels, 3, padding=1)
        )

    def forward(self, x, t, x_self_cond=None):
        if x_self_cond is None:
            x_self_cond = torch.zeros_like(x)
        
        x_in = torch.cat([x, x_self_cond], dim=1)
        h = self.input_conv(x_in)
        
        t_emb = self.t_proj(self.time_mlp(t))[:, :, None, None]
        h = h + t_emb
        
        for layer in self.layers:
            h = layer(h)
            
        return self.final_conv(h)

# ---------------- TRAIN & SAMPLE ----------------

def train_model(dataset, channels):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    model = FlowMatchingModel(channels).to(device)
    print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")

    opt = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for x1 in loader:
            x1 = x1.to(device)
            x0 = torch.randn_like(x1)
            t = torch.rand(x1.shape[0], device=device)
            t_res = t[:, None, None, None]
            xt = (1 - t_res) * x0 + t_res * x1
            
            # Self-conditioning: 50% chance to provide a null estimate
            x_self_cond = None
            if np.random.rand() > 0.5:
                with torch.no_grad():
                    # Estimate x1 from current velocity (v = x1 - x0 -> x1 = x0 + v)
                    v_pred = model(xt, t)
                    x_self_cond = xt + v_pred * (1.0 - t_res)

            pred_v = model(xt, t, x_self_cond)
            loss = loss_fn(pred_v, x1 - x0)

            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item()
        print(f"Epoch {epoch} | Loss {total_loss / len(loader):.6f}")
    return model

def sample_model(model, templates):
    model.eval()
    samples = []
    dt = 1.0 / SAMPLING_STEPS
    with torch.no_grad():
        for t_temp in templates:
            x = torch.randn_like(t_temp.unsqueeze(0)).to(device)
            x_self_cond = torch.zeros_like(x) # Initial null condition
            
            for i in range(SAMPLING_STEPS):
                t_val = i / SAMPLING_STEPS
                t_tensor = torch.full((1,), t_val, device=device)
                
                v = model(x, t_tensor, x_self_cond)
                x = x + v * dt
                # Update self-conditioning estimate of x1
                x_self_cond = x + v * (1.0 - t_val)
                
            samples.append(x.cpu().numpy())
    return samples

# ---------------- MAIN RUN ----------------

if __name__ == "__main__":
    with open(DATASET_PATH, "rb") as f:
        data = pickle.load(f)

    train_dataset = HypergraphTensorDataset(data["train"])
    test_tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H), 
                    train_dataset.max_N, train_dataset.max_E)) for H in data["test"]]
    
    channels = train_dataset.tensors[0].shape[0]
    metric_objs = [metrics.NodeNumDiff(),
                   metrics.NodeDegreeDistrWasserstein(),
                   metrics.EdgeSizeDistrWasserstein(),
                   metrics.Spectral(),
                   metrics.CentralityCloseness(),
                   metrics.CentralityBetweenness(),
                   metrics.CentralityHarmonic(),
                   metrics_mesh.ChamferNearestNeighborDistance(),
                   metrics_mesh.ChamferCoverage(),
                   metrics_mesh.ChamferDiversity()]

    all_metrics, all_generated = [], []

    for run in range(N_RUNS):
        print(f"\n--- RUN {run} ---")
        model = train_model(train_dataset, channels)
        gen_raw = sample_model(model, test_tensors)
        gen_hypergraphs = [tensor_to_hypergraph(t) for t in gen_raw]

        run_m = {str(m): m(data["test"], gen_hypergraphs, data["train"]) for m in metric_objs}
        for k, v in run_m.items(): print(f"{k}: {v:.4f}")
        all_metrics.append(run_m); all_generated.append(gen_hypergraphs)

    # Save logic
    with open(f"{OUTPUT_DIR}/generated_hypergraphs.pkl", "wb") as f: pickle.dump(all_generated, f)
    with open(f"{OUTPUT_DIR}/metrics_summary.txt", "w") as f:
        for k in all_metrics[0].keys():
            vals = [m[k] for m in all_metrics]
            f.write(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

    print(DATASET_PATH)
    for k in all_metrics[0].keys():
        vals = [m[k] for m in all_metrics]
        print(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

In [ ]:
import os
import pickle
import numpy as np
import math
from pathlib import Path
from typing import List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import hypernetx as hnx
import sys

# Paths
sys.path.append("/home/infres/gailhard/SODA_Hypergraph-Generation-Features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/SODA_Hypergraph-Generation-Features")

from metrics import metrics_hypergraph as metrics
from metrics import metrics_mesh as metrics_mesh

# ---------------- CONFIG ----------------

DATASET_PATH = "SODA_Hypergraph-Generation-Features/data/manifoldNetBench.pkl"
OUTPUT_DIR = "featured_flow_matching_self_cond"
N_RUNS = 5
EPOCHS = 50
BATCH_SIZE = 32 # Reduced batch size for larger model
LR = 1e-4
SAMPLING_STEPS = 25
FEATURE_DIM = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- UTILS ----------------

def shuffle_hypergraph_indices(incidence, node_features):
    N, E = incidence.shape
    node_perm = np.random.permutation(N)
    edge_perm = np.random.permutation(E)
    return incidence[node_perm][:, edge_perm], node_features[node_perm]

def hypergraph_to_tensor(H: hnx.Hypergraph, F_dim=FEATURE_DIM):
    incidence = H.incidence_matrix().toarray().astype(np.float32)
    node_features = np.stack([H.nodes[n].feature[:F_dim] for n in H.nodes], axis=0).astype(np.float32)
    incidence, node_features = shuffle_hypergraph_indices(incidence, node_features)
    N, E = incidence.shape
    tensor = np.zeros((1, N, E + F_dim), dtype=np.float32)
    tensor[0, :, :E], tensor[0, :, E:] = incidence, node_features
    return tensor

def tensor_to_hypergraph(tensor, F_dim=FEATURE_DIM, threshold=0.5):
    if tensor.ndim == 4: tensor = tensor[0]
    _, N, E_total = tensor.shape
    E = E_total - F_dim
    # Convert from [-1, 1] Tanh range to [0, 1] then threshold
    incidence_raw = (tensor[0, :, :E] + 1) / 2
    incidence = (incidence_raw > threshold).astype(int)
    
    node_features = tensor[0, :, E:] 
    hyperedges = {e: np.where(incidence[:, e] == 1)[0].tolist() for e in range(E) if np.any(incidence[:, e] == 1)}
    
    H = hnx.Hypergraph(hyperedges)
    for i in range(N): H.add_node(i, feature=node_features[i])
    if len(H.edges) == 0: H.add_edge(0, [0])
    return H

def pad_tensor(tensor, max_N, max_E):
    C, N, E = tensor.shape
    padded = np.zeros((C, max_N, max_E), dtype=np.float32)
    padded[:, :N, :E] = tensor
    return padded

class HypergraphTensorDataset(Dataset):
    def __init__(self, hypergraphs: list, F_dim=FEATURE_DIM):
        self.max_N = max(len(H.nodes) for H in hypergraphs)
        self.max_E = max(len(H.edges) for H in hypergraphs) + F_dim
        self.tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H, F_dim), self.max_N, self.max_E)) 
                        for H in hypergraphs for _ in range(10)]
    def __len__(self): return len(self.tensors)
    def __getitem__(self, idx): return self.tensors[idx]

# ---------------- SCALED MODEL (6M+ PARAMS) ----------------

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
        )
    def forward(self, x):
        return x + self.net(x)

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device) * -math.log(10000) / (half - 1))
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([args.sin(), args.cos()], dim=-1)
        return self.mlp(emb)

class FlowMatchingModel(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # channels * 2 because of self-conditioning input
        self.input_conv = nn.Conv2d(channels * 2, 128, 3, padding=1)
        self.time_mlp = TimeEmbedding(256)
        self.t_proj = nn.Linear(256, 128)

        # Bottleneck architecture to hit ~6M params
        self.layers = nn.ModuleList([
            ResBlock(128), ResBlock(128),
            nn.Conv2d(128, 256, 3, padding=1),
            ResBlock(256), ResBlock(256), ResBlock(256),
            nn.Conv2d(256, 128, 3, padding=1),
            ResBlock(128), ResBlock(128)
        ])
        
        self.final_conv = nn.Sequential(
            nn.GroupNorm(8, 128),
            nn.SiLU(),
            nn.Conv2d(128, channels, 3, padding=1)
        )

    def forward(self, x, t, x_self_cond=None):
        if x_self_cond is None:
            x_self_cond = torch.zeros_like(x)
        
        x_in = torch.cat([x, x_self_cond], dim=1)
        h = self.input_conv(x_in)
        
        t_emb = self.t_proj(self.time_mlp(t))[:, :, None, None]
        h = h + t_emb
        
        for layer in self.layers:
            h = layer(h)
            
        return self.final_conv(h)

# ---------------- TRAIN & SAMPLE ----------------

def train_model(dataset, channels):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    model = FlowMatchingModel(channels).to(device)
    print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")

    opt = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for x1 in loader:
            x1 = x1.to(device)
            x0 = torch.randn_like(x1)
            t = torch.rand(x1.shape[0], device=device)
            t_res = t[:, None, None, None]
            xt = (1 - t_res) * x0 + t_res * x1
            
            # Self-conditioning: 50% chance to provide a null estimate
            x_self_cond = None
            if np.random.rand() > 0.5:
                with torch.no_grad():
                    # Estimate x1 from current velocity (v = x1 - x0 -> x1 = x0 + v)
                    v_pred = model(xt, t)
                    x_self_cond = xt + v_pred * (1.0 - t_res)

            pred_v = model(xt, t, x_self_cond)
            loss = loss_fn(pred_v, x1 - x0)

            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item()
        print(f"Epoch {epoch} | Loss {total_loss / len(loader):.6f}")
    return model

def sample_model(model, templates):
    model.eval()
    samples = []
    dt = 1.0 / SAMPLING_STEPS
    with torch.no_grad():
        for t_temp in templates:
            x = torch.randn_like(t_temp.unsqueeze(0)).to(device)
            x_self_cond = torch.zeros_like(x) # Initial null condition
            
            for i in range(SAMPLING_STEPS):
                t_val = i / SAMPLING_STEPS
                t_tensor = torch.full((1,), t_val, device=device)
                
                v = model(x, t_tensor, x_self_cond)
                x = x + v * dt
                # Update self-conditioning estimate of x1
                x_self_cond = x + v * (1.0 - t_val)
                
            samples.append(x.cpu().numpy())
    return samples

# ---------------- MAIN RUN ----------------

if __name__ == "__main__":
    with open(DATASET_PATH, "rb") as f:
        data = pickle.load(f)

    train_dataset = HypergraphTensorDataset(data["train"])
    test_tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H), 
                    train_dataset.max_N, train_dataset.max_E)) for H in data["test"]]
    
    channels = train_dataset.tensors[0].shape[0]
    metric_objs = [metrics.NodeNumDiff(),
                   metrics.NodeDegreeDistrWasserstein(),
                   metrics.EdgeSizeDistrWasserstein(),
                   metrics.Spectral(),
                   metrics.CentralityCloseness(),
                   metrics.CentralityBetweenness(),
                   metrics.CentralityHarmonic(),
                   metrics_mesh.ChamferNearestNeighborDistance(),
                   metrics_mesh.ChamferCoverage(),
                   metrics_mesh.ChamferDiversity()]

    all_metrics, all_generated = [], []

    for run in range(N_RUNS):
        print(f"\n--- RUN {run} ---")
        model = train_model(train_dataset, channels)
        gen_raw = sample_model(model, test_tensors)
        gen_hypergraphs = [tensor_to_hypergraph(t) for t in gen_raw]

        run_m = {str(m): m(data["test"], gen_hypergraphs, data["train"]) for m in metric_objs}
        for k, v in run_m.items(): print(f"{k}: {v:.4f}")
        all_metrics.append(run_m); all_generated.append(gen_hypergraphs)

    # Save logic
    with open(f"{OUTPUT_DIR}/generated_hypergraphs.pkl", "wb") as f: pickle.dump(all_generated, f)
    with open(f"{OUTPUT_DIR}/metrics_summary.txt", "w") as f:
        for k in all_metrics[0].keys():
            vals = [m[k] for m in all_metrics]
            f.write(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

    print(DATASET_PATH)
    for k in all_metrics[0].keys():
        vals = [m[k] for m in all_metrics]
        print(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import hypernetx as hnx
import sys

# Paths
sys.path.append("/home/infres/gailhard/SODA_Hypergraph-Generation-Features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/SODA_Hypergraph-Generation-Features")

from metrics import metrics_hypergraph as metrics
from metrics import metrics_mesh as metrics_mesh

# ---------------- CONFIG ----------------

DATASET_PATH = "SODA_Hypergraph-Generation-Features/data/manifoldNetAirplane.pkl"
OUTPUT_DIR = "featured_hypergraph_wgan_gp_v2"
N_RUNS = 5
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-4
LATENT_DIM = 128
FEATURE_DIM = 3
LAMBDA_GP = 10
CRITIC_ITERATIONS = 5
SPARSITY_WEIGHT = 0.05 # L1 penalty to encourage sparse incidence matrices

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- UTILS ----------------

def shuffle_hypergraph_indices(incidence, node_features):
    N, E = incidence.shape
    node_perm = np.random.permutation(N)
    edge_perm = np.random.permutation(E)
    return incidence[node_perm][:, edge_perm], node_features[node_perm]

def hypergraph_to_tensor(H: hnx.Hypergraph, F_dim=FEATURE_DIM):
    incidence = H.incidence_matrix().toarray().astype(np.float32)
    node_features = np.stack([H.nodes[n].feature[:F_dim] for n in H.nodes], axis=0).astype(np.float32)
    incidence, node_features = shuffle_hypergraph_indices(incidence, node_features)
    N, E = incidence.shape
    tensor = np.zeros((1, N, E + F_dim), dtype=np.float32)
    tensor[0, :, :E], tensor[0, :, E:] = incidence, node_features
    return tensor

def tensor_to_hypergraph(tensor, F_dim=FEATURE_DIM, threshold=0.5):
    if tensor.ndim == 4: tensor = tensor[0]
    _, N, E_total = tensor.shape
    E = E_total - F_dim
    # Convert from [-1, 1] Tanh range to [0, 1] then threshold
    incidence_raw = (tensor[0, :, :E] + 1) / 2
    incidence = (incidence_raw > threshold).astype(int)
    
    node_features = tensor[0, :, E:] 
    hyperedges = {e: np.where(incidence[:, e] == 1)[0].tolist() for e in range(E) if np.any(incidence[:, e] == 1)}
    
    H = hnx.Hypergraph(hyperedges)
    for i in range(N): H.add_node(i, feature=node_features[i])
    if len(H.edges) == 0: H.add_edge(0, [0])
    return H

def pad_tensor(tensor, max_N, max_E):
    C, N, E = tensor.shape
    padded = np.zeros((C, max_N, max_E), dtype=np.float32)
    padded[:, :N, :E] = tensor
    return padded

class HypergraphTensorDataset(Dataset):
    def __init__(self, hypergraphs: list, F_dim=FEATURE_DIM):
        self.max_N = max(len(H.nodes) for H in hypergraphs)
        self.max_E = max(len(H.edges) for H in hypergraphs) + F_dim
        self.tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H, F_dim), self.max_N, self.max_E)) 
                        for H in hypergraphs for _ in range(10)]
    def __len__(self): return len(self.tensors)
    def __getitem__(self, idx): return self.tensors[idx]

# ---------------- MODELS ----------------

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
        )
    def forward(self, x): return x + self.net(x)

class Generator(nn.Module):
    def __init__(self, latent_dim, target_shape):
        super().__init__()
        self.target_shape = target_shape
        C, N, E = target_shape
        # Use PixelShuffle-friendly dimensions (must be divisible by 4)
        self.init_h, self.init_w = N // 4, E // 4
        
        self.l1 = nn.Linear(latent_dim, 512 * self.init_h * self.init_w)
        
        self.conv_blocks = nn.Sequential(
            nn.GroupNorm(32, 512),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.PixelShuffle(2), # Upscale 2x -> 64 channels
            ResBlock(64),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.PixelShuffle(2), # Upscale 2x -> 32 channels
            ResBlock(32),
            nn.Conv2d(32, C, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        out = self.l1(z).view(-1, 512, self.init_h, self.init_w)
        img = self.conv_blocks(out)
        return nn.functional.interpolate(img, size=(self.target_shape[1], self.target_shape[2]))

class Critic(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Increased filters significantly to match Generator capacity
        self.model = nn.Sequential(
            nn.Conv2d(channels, 128, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            ResBlock(256),
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            ResBlock(512),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        return self.model(x)

# ---------------- WGAN-GP LOGIC ----------------

def compute_gradient_penalty(critic, real_samples, fake_samples):
    alpha = torch.rand((real_samples.size(0), 1, 1, 1), device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = critic(interpolates)
    fake = torch.ones((real_samples.size(0), 1), device=device, requires_grad=False)
    
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

def train_wgan(dataset, channels):
    target_shape = dataset[0].shape
    gen = Generator(LATENT_DIM, target_shape).to(device)
    critic = Critic(channels).to(device)
    
    print(f"Generator Params: {sum(p.numel() for p in gen.parameters()):,}")
    print(f"Critic Params: {sum(p.numel() for p in critic.parameters()):,}")

    opt_G = optim.Adam(gen.parameters(), lr=LR, betas=(0.0, 0.9))
    opt_C = optim.Adam(critic.parameters(), lr=LR, betas=(0.0, 0.9))

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(EPOCHS):
        for i, real_imgs in enumerate(loader):
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)

            # --- Train Critic ---
            for _ in range(CRITIC_ITERATIONS):
                opt_C.zero_grad()
                z = torch.randn(batch_size, LATENT_DIM, device=device)
                fake_imgs = gen(z).detach()
                loss_C = critic(fake_imgs).mean() - critic(real_imgs).mean()
                gp = compute_gradient_penalty(critic, real_imgs, fake_imgs)
                (loss_C + LAMBDA_GP * gp).backward()
                opt_C.step()

            # --- Train Generator ---
            opt_G.zero_grad()
            gen_imgs = gen(z)
            # Wasserstein Loss + Sparsity Regularization (L1)
            loss_adv = -critic(gen_imgs).mean()
            loss_sparsity = torch.mean(torch.abs(gen_imgs[:, :, :, :target_shape[2]-FEATURE_DIM])) 
            
            (loss_adv + SPARSITY_WEIGHT * loss_sparsity).backward()
            opt_G.step()

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | C Loss: {loss_C.item():.4f} | G Loss Adv: {loss_adv.item():.4f} | Sparsity: {loss_sparsity.item():.4f}")

    return gen

# ---------------- EXPERIMENT RUNNER ----------------

if __name__ == "__main__":
    with open(DATASET_PATH, "rb") as f: data = pickle.load(f)
    train_ds = HypergraphTensorDataset(data["train"])
    channels = train_ds[0].shape[0]
    
    metric_objs = [metrics.NodeNumDiff(),
                   metrics.NodeDegreeDistrWasserstein(),
                   metrics.EdgeSizeDistrWasserstein(),
                   metrics.Spectral(),
                   metrics.CentralityCloseness(),
                   metrics.CentralityBetweenness(),
                   metrics.CentralityHarmonic(),
                   metrics_mesh.ChamferNearestNeighborDistance(),
                   metrics_mesh.ChamferCoverage(),
                   metrics_mesh.ChamferDiversity()]

    all_metrics, all_generated = [], []

    for run in range(N_RUNS):
        print(f"\n--- WGAN-GP RUN {run} ---")
        gen_model = train_wgan(train_ds, channels)
        
        gen_model.eval()
        with torch.no_grad():
            z = torch.randn(len(data["test"]), LATENT_DIM, device=device)
            samples = gen_model(z).cpu().numpy()
        
        gen_hypergraphs = [tensor_to_hypergraph(s) for s in samples]
        run_m = {str(m): m(data["test"], gen_hypergraphs, data["train"]) for m in metric_objs}
        
        for k, v in run_m.items(): print(f"{k}: {v:.4f}")
        all_metrics.append(run_m); all_generated.append(gen_hypergraphs)

    # Summary
    with open(f"{OUTPUT_DIR}/wgan_v2_summary.txt", "w") as f:
        for k in all_metrics[0].keys():
            vals = [m[k] for m in all_metrics]
            f.write(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

    print(DATASET_PATH)
    for k in all_metrics[0].keys():
        vals = [m[k] for m in all_metrics]
        print(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import hypernetx as hnx
import sys

# Paths
sys.path.append("/home/infres/gailhard/SODA_Hypergraph-Generation-Features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/SODA_Hypergraph-Generation-Features")

from metrics import metrics_hypergraph as metrics
from metrics import metrics_mesh as metrics_mesh

# ---------------- CONFIG ----------------

DATASET_PATH = "SODA_Hypergraph-Generation-Features/data/manifoldNetBench.pkl"
OUTPUT_DIR = "featured_hypergraph_wgan_gp_v2"
N_RUNS = 5
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-4
LATENT_DIM = 128
FEATURE_DIM = 3
LAMBDA_GP = 10
CRITIC_ITERATIONS = 5
SPARSITY_WEIGHT = 0.05 # L1 penalty to encourage sparse incidence matrices

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- UTILS ----------------

def shuffle_hypergraph_indices(incidence, node_features):
    N, E = incidence.shape
    node_perm = np.random.permutation(N)
    edge_perm = np.random.permutation(E)
    return incidence[node_perm][:, edge_perm], node_features[node_perm]

def hypergraph_to_tensor(H: hnx.Hypergraph, F_dim=FEATURE_DIM):
    incidence = H.incidence_matrix().toarray().astype(np.float32)
    node_features = np.stack([H.nodes[n].feature[:F_dim] for n in H.nodes], axis=0).astype(np.float32)
    incidence, node_features = shuffle_hypergraph_indices(incidence, node_features)
    N, E = incidence.shape
    tensor = np.zeros((1, N, E + F_dim), dtype=np.float32)
    tensor[0, :, :E], tensor[0, :, E:] = incidence, node_features
    return tensor

def tensor_to_hypergraph(tensor, F_dim=FEATURE_DIM, threshold=0.5):
    if tensor.ndim == 4: tensor = tensor[0]
    _, N, E_total = tensor.shape
    E = E_total - F_dim
    # Convert from [-1, 1] Tanh range to [0, 1] then threshold
    incidence_raw = (tensor[0, :, :E] + 1) / 2
    incidence = (incidence_raw > threshold).astype(int)
    
    node_features = tensor[0, :, E:] 
    hyperedges = {e: np.where(incidence[:, e] == 1)[0].tolist() for e in range(E) if np.any(incidence[:, e] == 1)}
    
    H = hnx.Hypergraph(hyperedges)
    for i in range(N): H.add_node(i, feature=node_features[i])
    if len(H.edges) == 0: H.add_edge(0, [0])
    return H

def pad_tensor(tensor, max_N, max_E):
    C, N, E = tensor.shape
    padded = np.zeros((C, max_N, max_E), dtype=np.float32)
    padded[:, :N, :E] = tensor
    return padded

class HypergraphTensorDataset(Dataset):
    def __init__(self, hypergraphs: list, F_dim=FEATURE_DIM):
        self.max_N = max(len(H.nodes) for H in hypergraphs)
        self.max_E = max(len(H.edges) for H in hypergraphs) + F_dim
        self.tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H, F_dim), self.max_N, self.max_E)) 
                        for H in hypergraphs for _ in range(10)]
    def __len__(self): return len(self.tensors)
    def __getitem__(self, idx): return self.tensors[idx]

# ---------------- MODELS ----------------

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
        )
    def forward(self, x): return x + self.net(x)

class Generator(nn.Module):
    def __init__(self, latent_dim, target_shape):
        super().__init__()
        self.target_shape = target_shape
        C, N, E = target_shape
        # Use PixelShuffle-friendly dimensions (must be divisible by 4)
        self.init_h, self.init_w = N // 4, E // 4
        
        self.l1 = nn.Linear(latent_dim, 512 * self.init_h * self.init_w)
        
        self.conv_blocks = nn.Sequential(
            nn.GroupNorm(32, 512),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.PixelShuffle(2), # Upscale 2x -> 64 channels
            ResBlock(64),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.PixelShuffle(2), # Upscale 2x -> 32 channels
            ResBlock(32),
            nn.Conv2d(32, C, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        out = self.l1(z).view(-1, 512, self.init_h, self.init_w)
        img = self.conv_blocks(out)
        return nn.functional.interpolate(img, size=(self.target_shape[1], self.target_shape[2]))

class Critic(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Increased filters significantly to match Generator capacity
        self.model = nn.Sequential(
            nn.Conv2d(channels, 128, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            ResBlock(256),
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            ResBlock(512),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        return self.model(x)

# ---------------- WGAN-GP LOGIC ----------------

def compute_gradient_penalty(critic, real_samples, fake_samples):
    alpha = torch.rand((real_samples.size(0), 1, 1, 1), device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = critic(interpolates)
    fake = torch.ones((real_samples.size(0), 1), device=device, requires_grad=False)
    
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

def train_wgan(dataset, channels):
    target_shape = dataset[0].shape
    gen = Generator(LATENT_DIM, target_shape).to(device)
    critic = Critic(channels).to(device)
    
    print(f"Generator Params: {sum(p.numel() for p in gen.parameters()):,}")
    print(f"Critic Params: {sum(p.numel() for p in critic.parameters()):,}")

    opt_G = optim.Adam(gen.parameters(), lr=LR, betas=(0.0, 0.9))
    opt_C = optim.Adam(critic.parameters(), lr=LR, betas=(0.0, 0.9))

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(EPOCHS):
        for i, real_imgs in enumerate(loader):
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)

            # --- Train Critic ---
            for _ in range(CRITIC_ITERATIONS):
                opt_C.zero_grad()
                z = torch.randn(batch_size, LATENT_DIM, device=device)
                fake_imgs = gen(z).detach()
                loss_C = critic(fake_imgs).mean() - critic(real_imgs).mean()
                gp = compute_gradient_penalty(critic, real_imgs, fake_imgs)
                (loss_C + LAMBDA_GP * gp).backward()
                opt_C.step()

            # --- Train Generator ---
            opt_G.zero_grad()
            gen_imgs = gen(z)
            # Wasserstein Loss + Sparsity Regularization (L1)
            loss_adv = -critic(gen_imgs).mean()
            loss_sparsity = torch.mean(torch.abs(gen_imgs[:, :, :, :target_shape[2]-FEATURE_DIM])) 
            
            (loss_adv + SPARSITY_WEIGHT * loss_sparsity).backward()
            opt_G.step()

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | C Loss: {loss_C.item():.4f} | G Loss Adv: {loss_adv.item():.4f} | Sparsity: {loss_sparsity.item():.4f}")

    return gen

# ---------------- EXPERIMENT RUNNER ----------------

if __name__ == "__main__":
    with open(DATASET_PATH, "rb") as f: data = pickle.load(f)
    train_ds = HypergraphTensorDataset(data["train"])
    channels = train_ds[0].shape[0]
    
    metric_objs = [metrics.NodeNumDiff(),
                   metrics.NodeDegreeDistrWasserstein(),
                   metrics.EdgeSizeDistrWasserstein(),
                   metrics.Spectral(),
                   metrics.CentralityCloseness(),
                   metrics.CentralityBetweenness(),
                   metrics.CentralityHarmonic(),
                   metrics_mesh.ChamferNearestNeighborDistance(),
                   metrics_mesh.ChamferCoverage(),
                   metrics_mesh.ChamferDiversity()]

    all_metrics, all_generated = [], []

    for run in range(N_RUNS):
        print(f"\n--- WGAN-GP RUN {run} ---")
        gen_model = train_wgan(train_ds, channels)
        
        gen_model.eval()
        with torch.no_grad():
            z = torch.randn(len(data["test"]), LATENT_DIM, device=device)
            samples = gen_model(z).cpu().numpy()
        
        gen_hypergraphs = [tensor_to_hypergraph(s) for s in samples]
        run_m = {str(m): m(data["test"], gen_hypergraphs, data["train"]) for m in metric_objs}
        
        for k, v in run_m.items(): print(f"{k}: {v:.4f}")
        all_metrics.append(run_m); all_generated.append(gen_hypergraphs)

    # Summary
    with open(f"{OUTPUT_DIR}/wgan_v2_summary.txt", "w") as f:
        for k in all_metrics[0].keys():
            vals = [m[k] for m in all_metrics]
            f.write(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

    print(DATASET_PATH)
    for k in all_metrics[0].keys():
        vals = [m[k] for m in all_metrics]
        print(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import hypernetx as hnx
import sys
import math

# Paths
sys.path.append("/home/infres/gailhard/SODA_Hypergraph-Generation-Features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/SODA_Hypergraph-Generation-Features")

from metrics import metrics_hypergraph as metrics
from metrics import metrics_mesh as metrics_mesh

# ---------------- CONFIG ----------------

DATASET_PATH = "SODA_Hypergraph-Generation-Features/data/manifoldNetAirplane.pkl"
OUTPUT_DIR = "baseline_vae_corrected"
N_RUNS = 5
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-4
LATENT_DIM = 256
FEATURE_DIM = 3
KL_WEIGHT = 0.001 
ANNEAL_EPOCHS = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- UTILS ----------------

def shuffle_hypergraph_indices(incidence, node_features):
    N, E = incidence.shape
    node_perm = np.random.permutation(N)
    edge_perm = np.random.permutation(E)
    return incidence[node_perm][:, edge_perm], node_features[node_perm]

def hypergraph_to_tensor(H, F_dim=FEATURE_DIM):
    incidence = H.incidence_matrix().toarray().astype(np.float32)
    node_features = np.stack([H.nodes[n].feature[:F_dim] for n in H.nodes], axis=0).astype(np.float32)
    incidence, node_features = shuffle_hypergraph_indices(incidence, node_features)
    N, E = incidence.shape
    tensor = np.zeros((1, N, E + F_dim), dtype=np.float32)
    tensor[0, :, :E], tensor[0, :, E:] = incidence, node_features
    return tensor

def tensor_to_hypergraph(tensor, F_dim=FEATURE_DIM, threshold=0.5):
    if tensor.ndim == 4: tensor = tensor[0]
    _, N, E_total = tensor.shape
    E = E_total - F_dim
    # Corrected: VAE uses Sigmoid, so input is already [0, 1]
    incidence = (tensor[0, :, :E] > threshold).astype(int)
    node_features = tensor[0, :, E:] 
    
    hyperedges = {e: np.where(incidence[:, e] == 1)[0].tolist() 
                  for e in range(E) if np.any(incidence[:, e] == 1)}
    
    H = hnx.Hypergraph(hyperedges)
    for i in range(N):
        H.add_node(i, feature=node_features[i])
    if len(H.edges) == 0: H.add_edge(0, [0])
    return H

def pad_tensor(tensor, max_N, max_E):
    C, N, E = tensor.shape
    padded = np.zeros((C, max_N, max_E), dtype=np.float32)
    padded[:, :N, :E] = tensor
    return padded

class HypergraphTensorDataset(Dataset):
    def __init__(self, hypergraphs, F_dim=FEATURE_DIM):
        self.max_N = max(len(H.nodes) for H in hypergraphs)
        self.F = F_dim
        self.max_E = max(len(H.edges) for H in hypergraphs) + self.F
        self.tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H, self.F), self.max_N, self.max_E)) 
                        for H in hypergraphs for _ in range(10)]
    def __len__(self): return len(self.tensors)
    def __getitem__(self, idx): return self.tensors[idx]

# ---------------- MODELS ----------------

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
        )
    def forward(self, x): return x + self.net(x)

class Encoder(nn.Module):
    def __init__(self, channels, latent_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(channels, 64, 3, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.SiLU(),
            ResBlock(128),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)

    def forward(self, x):
        h = self.model(x)
        return self.fc_mu(h), self.fc_logvar(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim, target_shape):
        super().__init__()
        self.target_shape = target_shape
        C, N, E = target_shape
        self.init_h, self.init_w = N // 4, E // 4
        
        self.l1 = nn.Linear(latent_dim, 256 * self.init_h * self.init_w)
        self.conv_blocks = nn.Sequential(
            nn.GroupNorm(32, 256),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.PixelShuffle(2),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.PixelShuffle(2),
            ResBlock(16),
            nn.Conv2d(16, C, 3, padding=1),
            nn.Sigmoid() 
        )

    def forward(self, z):
        out = self.l1(z).view(-1, 256, self.init_h, self.init_w)
        img = self.conv_blocks(out)
        return nn.functional.interpolate(img, size=(self.target_shape[1], self.target_shape[2]))

class VAE(nn.Module):
    def __init__(self, channels, latent_dim, target_shape):
        super().__init__()
        self.encoder = Encoder(channels, latent_dim)
        self.decoder = Decoder(latent_dim, target_shape)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

# ---------------- TRAINING ----------------

def train_vae(dataset, channels):
    target_shape = dataset[0].shape
    E_dim = target_shape[2] - FEATURE_DIM
    model = VAE(channels, LATENT_DIM, target_shape).to(device)
    print(f"Model Params: {sum(p.numel() for p in model.parameters()):,}")
    
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(EPOCHS):
        beta = min(KL_WEIGHT, KL_WEIGHT * (epoch / ANNEAL_EPOCHS))
        total_loss = 0
        
        for x in loader:
            x = x.to(device)
            recon_x, mu, logvar = model(x)
            
            # Binary Cross Entropy for incidence, MSE for features
            recon_inc = nn.BCELoss(reduction='sum')(recon_x[:, :, :, :E_dim], x[:, :, :, :E_dim])
            recon_feat = nn.MSELoss(reduction='sum')(recon_x[:, :, :, E_dim:], x[:, :, :, E_dim:])
            
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            
            # Combined Loss
            loss = (recon_inc + recon_feat + beta * kl_loss) / x.size(0)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Beta: {beta:.6f} | Avg Loss: {total_loss/len(loader):.2f}")
    return model

# ---------------- EXPERIMENT ----------------

if __name__ == "__main__":
    with open(DATASET_PATH, "rb") as f: 
        data = pickle.load(f)
    
    train_hypergraphs = data["train"]
    test_hypergraphs = data["test"]
    
    train_ds = HypergraphTensorDataset(train_hypergraphs)
    channels = train_ds[0].shape[0]
    
    metric_objs = [metrics.NodeNumDiff(),
                   metrics.NodeDegreeDistrWasserstein(),
                   metrics.EdgeSizeDistrWasserstein(),
                   metrics.Spectral(),
                   metrics.CentralityCloseness(),
                   metrics.CentralityBetweenness(),
                   metrics.CentralityHarmonic(),
                   metrics_mesh.ChamferNearestNeighborDistance(),
                   metrics_mesh.ChamferCoverage(),
                   metrics_mesh.ChamferDiversity()]

    for run in range(N_RUNS):
        print(f"\n--- VAE RUN {run} ---")
        vae_model = train_vae(train_ds, channels)
        vae_model.eval()
        
        with torch.no_grad():
            z = torch.randn(len(test_hypergraphs), LATENT_DIM, device=device)
            samples = vae_model.decoder(z).cpu().numpy()
        
        gen_hypergraphs = [tensor_to_hypergraph(s) for s in samples]
        run_m = {str(m): m(data["test"], gen_hypergraphs, data["train"]) for m in metric_objs}
        
        for k, v in run_m.items(): print(f"{k}: {v:.4f}")
        all_metrics.append(run_m); all_generated.append(gen_hypergraphs)

    # Summary
    with open(f"{OUTPUT_DIR}/wgan_v2_summary.txt", "w") as f:
        for k in all_metrics[0].keys():
            vals = [m[k] for m in all_metrics]
            f.write(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

    print(DATASET_PATH)
    for k in all_metrics[0].keys():
        vals = [m[k] for m in all_metrics]
        print(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import hypernetx as hnx
import sys
import math

# Paths
sys.path.append("/home/infres/gailhard/SODA_Hypergraph-Generation-Features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/SODA_Hypergraph-Generation-Features")

from metrics import metrics_hypergraph as metrics
from metrics import metrics_mesh as metrics_mesh

# ---------------- CONFIG ----------------

DATASET_PATH = "SODA_Hypergraph-Generation-Features/data/manifoldNetBench.pkl"
OUTPUT_DIR = "baseline_vae_corrected"
N_RUNS = 5
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-4
LATENT_DIM = 256
FEATURE_DIM = 3
KL_WEIGHT = 0.001 
ANNEAL_EPOCHS = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- UTILS ----------------

def shuffle_hypergraph_indices(incidence, node_features):
    N, E = incidence.shape
    node_perm = np.random.permutation(N)
    edge_perm = np.random.permutation(E)
    return incidence[node_perm][:, edge_perm], node_features[node_perm]

def hypergraph_to_tensor(H, F_dim=FEATURE_DIM):
    incidence = H.incidence_matrix().toarray().astype(np.float32)
    node_features = np.stack([H.nodes[n].feature[:F_dim] for n in H.nodes], axis=0).astype(np.float32)
    incidence, node_features = shuffle_hypergraph_indices(incidence, node_features)
    N, E = incidence.shape
    tensor = np.zeros((1, N, E + F_dim), dtype=np.float32)
    tensor[0, :, :E], tensor[0, :, E:] = incidence, node_features
    return tensor

def tensor_to_hypergraph(tensor, F_dim=FEATURE_DIM, threshold=0.5):
    if tensor.ndim == 4: tensor = tensor[0]
    _, N, E_total = tensor.shape
    E = E_total - F_dim
    # Corrected: VAE uses Sigmoid, so input is already [0, 1]
    incidence = (tensor[0, :, :E] > threshold).astype(int)
    node_features = tensor[0, :, E:] 
    
    hyperedges = {e: np.where(incidence[:, e] == 1)[0].tolist() 
                  for e in range(E) if np.any(incidence[:, e] == 1)}
    
    H = hnx.Hypergraph(hyperedges)
    for i in range(N):
        H.add_node(i, feature=node_features[i])
    if len(H.edges) == 0: H.add_edge(0, [0])
    return H

def pad_tensor(tensor, max_N, max_E):
    C, N, E = tensor.shape
    padded = np.zeros((C, max_N, max_E), dtype=np.float32)
    padded[:, :N, :E] = tensor
    return padded

class HypergraphTensorDataset(Dataset):
    def __init__(self, hypergraphs, F_dim=FEATURE_DIM):
        self.max_N = max(len(H.nodes) for H in hypergraphs)
        self.F = F_dim
        self.max_E = max(len(H.edges) for H in hypergraphs) + self.F
        self.tensors = [torch.from_numpy(pad_tensor(hypergraph_to_tensor(H, self.F), self.max_N, self.max_E)) 
                        for H in hypergraphs for _ in range(10)]
    def __len__(self): return len(self.tensors)
    def __getitem__(self, idx): return self.tensors[idx]

# ---------------- MODELS ----------------

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
        )
    def forward(self, x): return x + self.net(x)

class Encoder(nn.Module):
    def __init__(self, channels, latent_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(channels, 64, 3, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.SiLU(),
            ResBlock(128),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)

    def forward(self, x):
        h = self.model(x)
        return self.fc_mu(h), self.fc_logvar(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim, target_shape):
        super().__init__()
        self.target_shape = target_shape
        C, N, E = target_shape
        self.init_h, self.init_w = N // 4, E // 4
        
        self.l1 = nn.Linear(latent_dim, 256 * self.init_h * self.init_w)
        self.conv_blocks = nn.Sequential(
            nn.GroupNorm(32, 256),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.PixelShuffle(2),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.PixelShuffle(2),
            ResBlock(16),
            nn.Conv2d(16, C, 3, padding=1),
            nn.Sigmoid() 
        )

    def forward(self, z):
        out = self.l1(z).view(-1, 256, self.init_h, self.init_w)
        img = self.conv_blocks(out)
        return nn.functional.interpolate(img, size=(self.target_shape[1], self.target_shape[2]))

class VAE(nn.Module):
    def __init__(self, channels, latent_dim, target_shape):
        super().__init__()
        self.encoder = Encoder(channels, latent_dim)
        self.decoder = Decoder(latent_dim, target_shape)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

# ---------------- TRAINING ----------------

def train_vae(dataset, channels):
    target_shape = dataset[0].shape
    E_dim = target_shape[2] - FEATURE_DIM
    model = VAE(channels, LATENT_DIM, target_shape).to(device)
    print(f"Model Params: {sum(p.numel() for p in model.parameters()):,}")
    
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(EPOCHS):
        beta = min(KL_WEIGHT, KL_WEIGHT * (epoch / ANNEAL_EPOCHS))
        total_loss = 0
        
        for x in loader:
            x = x.to(device)
            recon_x, mu, logvar = model(x)
            
            # Binary Cross Entropy for incidence, MSE for features
            recon_inc = nn.BCELoss(reduction='sum')(recon_x[:, :, :, :E_dim], x[:, :, :, :E_dim])
            recon_feat = nn.MSELoss(reduction='sum')(recon_x[:, :, :, E_dim:], x[:, :, :, E_dim:])
            
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            
            # Combined Loss
            loss = (recon_inc + recon_feat + beta * kl_loss) / x.size(0)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Beta: {beta:.6f} | Avg Loss: {total_loss/len(loader):.2f}")
    return model

# ---------------- EXPERIMENT ----------------

if __name__ == "__main__":
    with open(DATASET_PATH, "rb") as f: 
        data = pickle.load(f)
    
    train_hypergraphs = data["train"]
    test_hypergraphs = data["test"]
    
    train_ds = HypergraphTensorDataset(train_hypergraphs)
    channels = train_ds[0].shape[0]
    
    metric_objs = [metrics.NodeNumDiff(),
                   metrics.NodeDegreeDistrWasserstein(),
                   metrics.EdgeSizeDistrWasserstein(),
                   metrics.Spectral(),
                   metrics.CentralityCloseness(),
                   metrics.CentralityBetweenness(),
                   metrics.CentralityHarmonic(),
                   metrics_mesh.ChamferNearestNeighborDistance(),
                   metrics_mesh.ChamferCoverage(),
                   metrics_mesh.ChamferDiversity()]

    for run in range(N_RUNS):
        print(f"\n--- VAE RUN {run} ---")
        vae_model = train_vae(train_ds, channels)
        vae_model.eval()
        
        with torch.no_grad():
            z = torch.randn(len(test_hypergraphs), LATENT_DIM, device=device)
            samples = vae_model.decoder(z).cpu().numpy()
        
        gen_hypergraphs = [tensor_to_hypergraph(s) for s in samples]
        run_m = {str(m): m(data["test"], gen_hypergraphs, data["train"]) for m in metric_objs}
        
        for k, v in run_m.items(): print(f"{k}: {v:.4f}")
        all_metrics.append(run_m); all_generated.append(gen_hypergraphs)

    # Summary
    with open(f"{OUTPUT_DIR}/wgan_v2_summary.txt", "w") as f:
        for k in all_metrics[0].keys():
            vals = [m[k] for m in all_metrics]
            f.write(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

    print(DATASET_PATH)
    for k in all_metrics[0].keys():
        vals = [m[k] for m in all_metrics]
        print(f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}\n")

## Point clouds

In [ ]:
import os
import pickle
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import networkx as nx
import sys

# ---------------- CONFIG ----------------

# Paths
sys.path.append("/home/infres/gailhard/graph-generation-features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/graph-generation-features")

import metrics

DATASET_PATH = "/home/infres/gailhard/graph-generation-features/data/pointCloudAirplane.pkl"
OUTPUT_DIR = "graph_pointcloud_flow_matching_output"
N_RUNS = 5
EPOCHS = 50
BATCH_SIZE = 1
LR = 1e-4
SAMPLING_STEPS = 25
POS_DIM = 3 # x, y, z positions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- UTILS ----------------

def shuffle_graph_indices(adj, positions):
    N = adj.shape[0]
    perm = np.random.permutation(N)
    adj = adj[perm][:, perm]
    positions = positions[perm]
    return adj, positions

def graph_to_tensor(G: nx.Graph, max_N: int):
    """Converts a NetworkX graph with 'feature' attributes to a (1, N, N+3) tensor."""
    adj = nx.to_numpy_array(G).astype(np.float32)
    
    # Extract 3D positions from nodes
    positions = np.array([G.nodes[n]['feature'] for n in G.nodes]).astype(np.float32)
    
    # Node shuffling for permutation invariance
    adj, positions = shuffle_graph_indices(adj, positions)
    
    N = adj.shape[0]
    # Tensor shape: (1, N, N + 3) 
    tensor = np.zeros((1, N, N + POS_DIM), dtype=np.float32)
    tensor[0, :, :N] = adj
    tensor[0, :, N:] = positions
    return tensor

def tensor_to_graph(tensor, threshold=0.5):
    """Reconstructs the NetworkX graph and point cloud from the generated tensor."""
    if tensor.ndim == 4: 
        tensor = tensor[0]
    
    _, N, total_cols = tensor.shape
    
    # Extract adjacency and positions
    adj_raw = tensor[0, :, :N]
    pos_raw = tensor[0, :, N:]
    
    # Thresholding for discrete edges
    adj = (adj_raw > threshold).astype(int)
    G = nx.from_numpy_array(adj)
    
    # Assign reconstructed 3D positions
    for i, p in enumerate(pos_raw):
        if i < len(G.nodes):
            G.nodes[i]['feature'] = p
            
    return G

def pad_graph_tensor(tensor, max_N):
    """Pads adjacency matrix to (max_N, max_N) while keeping positions at the end."""
    C, N, _ = tensor.shape
    padded = np.zeros((C, max_N, max_N + POS_DIM), dtype=np.float32)
    
    padded[:, :N, :N] = tensor[:, :, :N]
    padded[:, :N, max_N:] = tensor[:, :, N:]
    return padded

# ---------------- DATASET ----------------

class GraphTensorDataset(Dataset):
    def __init__(self, graphs: list):
        self.max_N = max(len(G.nodes) for G in graphs)
        # Augmenting the dataset 10x with different shuffles
        self.tensors = [
            torch.from_numpy(pad_graph_tensor(graph_to_tensor(G, self.max_N), self.max_N))
            for G in graphs for _ in range(10)
        ]

    def __len__(self): return len(self.tensors)
    def __getitem__(self, idx): return self.tensors[idx]

# ---------------- MODEL ----------------

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(8, dim),
        )
    def forward(self, x): return x + self.net(x)

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device) * -math.log(10000) / (half - 1))
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([args.sin(), args.cos()], dim=-1)
        return self.mlp(emb)

class FlowMatchingModel(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.input_conv = nn.Conv2d(channels * 2, 128, 3, padding=1)
        self.time_mlp = TimeEmbedding(256)
        self.t_proj = nn.Linear(256, 128)

        self.layers = nn.ModuleList([
            ResBlock(128), ResBlock(128),
            nn.Conv2d(128, 256, 3, padding=1),
            ResBlock(256), ResBlock(256),
            nn.Conv2d(256, 128, 3, padding=1),
            ResBlock(128), ResBlock(128)
        ])
        
        self.final_conv = nn.Sequential(
            nn.GroupNorm(8, 128),
            nn.SiLU(),
            nn.Conv2d(128, channels, 3, padding=1)
        )

    def forward(self, x, t, x_self_cond=None):
        if x_self_cond is None:
            x_self_cond = torch.zeros_like(x)
        
        x_in = torch.cat([x, x_self_cond], dim=1)
        h = self.input_conv(x_in)
        
        t_emb = self.t_proj(self.time_mlp(t))[:, :, None, None]
        h = h + t_emb
        
        for layer in self.layers:
            h = layer(h)
            
        return self.final_conv(h)

# ---------------- TRAIN & SAMPLE ----------------

def train_model(dataset, channels):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    model = FlowMatchingModel(channels).to(device)
    
    # Print total model parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total model parameters: {total_params:,}")

    opt = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for x1 in loader:
            x1 = x1.to(device)
            x0 = torch.randn_like(x1)
            t = torch.rand(x1.shape[0], device=device)
            t_res = t[:, None, None, None]
            
            # Linear Interpolation (Optimal Transport Path)
            xt = (1 - t_res) * x0 + t_res * x1
            
            x_self_cond = None
            if np.random.rand() > 0.5:
                with torch.no_grad():
                    v_pred = model(xt, t)
                    x_self_cond = xt + v_pred * (1.0 - t_res)

            pred_v = model(xt, t, x_self_cond)
            loss = loss_fn(pred_v, x1 - x0)

            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item()
            
        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Loss {total_loss / len(loader):.6f}")
    return model

def sample_model(model, templates):
    model.eval()
    samples = []
    dt = 1.0 / SAMPLING_STEPS
    with torch.no_grad():
        for t_temp in templates:
            x = torch.randn_like(t_temp.unsqueeze(0)).to(device)
            x_self_cond = torch.zeros_like(x)
            
            for i in range(SAMPLING_STEPS):
                t_val = i / SAMPLING_STEPS
                t_tensor = torch.full((1,), t_val, device=device)
                
                v = model(x, t_tensor, x_self_cond)
                x = x + v * dt
                x_self_cond = x + v * (1.0 - t_val)
                
            samples.append(x.cpu().numpy())
    return samples

# ---------------- MAIN RUN ----------------

if __name__ == "__main__":
    with open(DATASET_PATH, "rb") as f:
        data = pickle.load(f)

    train_graphs = data["train"]
    test_graphs = data["test"]

    train_dataset = GraphTensorDataset(train_graphs)
    test_tensors = [
        torch.from_numpy(pad_graph_tensor(graph_to_tensor(G, train_dataset.max_N), train_dataset.max_N))
        for G in test_graphs
    ]
    
    channels = train_dataset.tensors[0].shape[0]

    # Add your specific metrics to this list
    metric_objs = [
           metrics.NodeNumDiff(),
           metrics.NodeDegreeDistrWasserstein(),
           metrics.EdgeSizeDistrWasserstein(),
           metrics.Spectral(),
           metrics.CentralityCloseness(),
           metrics.CentralityBetweenness(),
           metrics.CentralityHarmonic(),
           metrics.ChamferNearestNeighborDistance(),
           metrics.ChamferCoverage(),
           metrics.ChamferDiversity()
    ]

    all_metrics = []
    all_generated = []

    for run in range(N_RUNS):
        print(f"\n================ RUN {run} ================")
        model = train_model(train_dataset, channels)
        
        # Generation phase
        gen_raw = sample_model(model, test_tensors)
        gen_graphs = [tensor_to_graph(t) for t in gen_raw]

        # Metric Evaluation
        run_m = {}
        for m in metric_objs:
            # Note: Ensure your metric classes support nx.Graph input if you changed them
            val = m(test_graphs, gen_graphs, train_graphs)
            run_m[str(m)] = val
            print(f"{m}: {val:.4f}")
        
        all_metrics.append(run_m)
        all_generated.append(gen_graphs)

    # ---------------- SAVE & SUMMARY ----------------

    # Save all generated graphs for this run
    with open(f"{OUTPUT_DIR}/generated_pointclouds.pkl", "wb") as f:
        pickle.dump(all_generated, f)

    # Write summary stats to file
    summary_path = f"{OUTPUT_DIR}/metrics_summary.txt"
    with open(summary_path, "w") as f:
        f.write(f"Dataset: {DATASET_PATH}\n")
        keys = all_metrics[0].keys()
        for k in keys:
            vals = [run[k] for run in all_metrics]
            mean, std = np.mean(vals), np.std(vals)
            f.write(f"{k}: mean={mean:.4f}, std={std:.4f}\n")
            # Also print to console
            print(f"{k}: mean={mean:.4f}, std={std:.4f}")

    print(f"\nAll results saved to: {OUTPUT_DIR}")

## Sequential

In [ ]:
import os
import sys
import pickle
import random
from pathlib import Path

import numpy as np
import scipy as sp
import torch as th
import torch.nn as nn
import torch.optim as optim

import warnings
import numpy as np
from scipy.sparse import coo_matrix, csgraph, linalg

from scipy.sparse import coo_array, csr_array, hstack, vstack
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch

# ---------------- PATHS / CONFIG ----------------

sys.path.append("/home/infres/gailhard/SODA_Hypergraph-Generation-Features/graph_generation")
sys.path.insert(0, "/home/infres/gailhard/SODA_Hypergraph-Generation-Features")

import graph_generation as gg

from metrics import metrics_hypergraph as metrics
from metrics import metrics_mesh as metrics_mesh

DATASET_PATH = "/home/infres/gailhard/SODA_Hypergraph-Generation-Features/data/manifoldNetAirplane.pkl"

GENERATED_TOPOLOGIES_PATHS = [
    "/home/infres/gailhard/SODA_Hypergraph-Generation-Features/outputs/2026-05-27/13-02-27/test/step_700000.pkl",
    "/home/infres/gailhard/SODA_Hypergraph-Generation-Features/outputs/2026-05-27/13-02-08/test/step_700000.pkl",
    "/home/infres/gailhard/SODA_Hypergraph-Generation-Features/outputs/2026-05-27/12-59-08/test/step_500000.pkl",
]

OUTPUT_DIR = "sparse_ppgn_position_diffusion_many_topology_pkls"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 200
BATCH_SIZE = 32
LR = 1e-4
SAMPLING_STEPS = 256

POS_DIM = 3
NUM_EIGENVECTORS = 8

DEVICE = th.device("cuda" if th.cuda.is_available() else "cpu")
print("Device:", DEVICE)

MODEL_CFG = dict(
    node_features_emb_dim=128,
    hyperedge_features_emb_dim=128,
    expansion_emb_dim=32,
    hidden_dim=256,
    ppgn_dim=128,
    num_layers=10,
    dropout=0.0,
)

SIGN_NET_CFG = dict(
    hidden_dim=128,
    num_layers=5,
)

# ---------------- HYPERGRAPH CONVERSION ----------------

def getIncidenceMatrix(H) -> csr_array:
    node_list = list(H.nodes)
    edge_list = list(H.edges)

    dic_nodes = {node: i for i, node in enumerate(node_list)}
    dic_edges = {edge: i for i, edge in enumerate(edge_list)}

    num_entries = sum(len(H.edges[edge]) for edge in edge_list)
    rows = np.empty(num_entries, dtype=int)
    cols = np.empty(num_entries, dtype=int)

    idx = 0
    for edge in edge_list:
        for node in H.edges[edge]:
            rows[idx] = dic_nodes[node]
            cols[idx] = dic_edges[edge]
            idx += 1

    data = np.ones(num_entries, dtype=int)
    coo_mat = coo_array((data, (rows, cols)), shape=(len(node_list), len(edge_list)))
    
    return csr_array(coo_mat)


def getBipartiteAdj(incidence_matrix) -> csr_array:
    # Ensure the incidence matrix is in CSR format
    incidence_matrix = csr_array(incidence_matrix)
    
    # Number of nodes and hyperedges
    num_nodes, num_hyperedges = incidence_matrix.shape
    
    # Create zero matrices for the non-connected parts
    zero_nodes = csr_array((num_nodes, num_nodes))
    zero_hyperedges = csr_array((num_hyperedges, num_hyperedges))
    
    # Stack and concatenate the matrices to form the bipartite adjacency matrix
    top = hstack([zero_nodes, incidence_matrix])
    bottom = hstack([incidence_matrix.T, zero_hyperedges])
    bipartite_adjacency = vstack([top, bottom])
    
    return csr_array(bipartite_adjacency)


def compute_spectral_features(bipartite_adj, k):
    n = bipartite_adj.shape[0]
    adj = coo_matrix(bipartite_adj, copy=True, dtype=np.float64)
    lap = csgraph.laplacian(adj, normed=True, copy=False)

    eigenvalues_full = np.zeros(k)
    eigenvectors_full = np.zeros((n, k))

    k_eff = min(n - 2, k)

    if k_eff > 0:
        while True:
            try:
                eigenvalues, eigenvectors = linalg.eigsh(
                    lap,
                    k=k_eff + 1,
                    which="SM",
                )
                break
            except Exception:
                warnings.warn("eigsh failed to converge, trying again")

        eigenvalues_full[:k_eff] = eigenvalues[1:]
        eigenvectors_full[:, :k_eff] = eigenvectors[:, 1:]

    eigenvalues_repeated = eigenvalues_full[np.newaxis, :].repeat(n, axis=0)

    spectral_features = np.concatenate(
        (eigenvalues_repeated, eigenvectors_full),
        axis=1,
    ).astype(np.float32)

    return th.from_numpy(spectral_features)


def hypergraph_to_data(H, extract_node_features):
    incidence = getIncidenceMatrix(H).tocoo()
    bipartite_adj = getBipartiteAdj(incidence).tocoo()
    
    num_real_nodes = incidence.shape[0]
    num_hyperedges = incidence.shape[1]
    total_nodes = num_real_nodes + num_hyperedges
    
    edge_index = th.tensor(
        np.stack([bipartite_adj.row, bipartite_adj.col], axis=0),
        dtype=th.long,
    )
    
    node_type = th.cat(
        [
            th.ones(num_real_nodes, dtype=th.long),
            th.zeros(num_hyperedges, dtype=th.long),
        ],
        dim=0,
    )

    if extract_node_features:
        node_features = th.as_tensor([H.nodes[n].feature for n in H.nodes], dtype=th.float32)
    else:
        node_features = None

    spectral_features = compute_spectral_features(
        bipartite_adj,
        NUM_EIGENVECTORS,
    )

    data = Data(
        edge_index=edge_index,

        node_type=node_type,
        budget=th.ones(num_real_nodes, dtype=th.float32),

        node_features=node_features,
        node_features_initial=th.zeros(num_real_nodes, POS_DIM, dtype=th.float32),

        spectral_features=spectral_features,

        red_frac=th.zeros(1, dtype=th.float32),
        noise_cond=th.zeros(1, dtype=th.float32),
        target_size=th.tensor([num_real_nodes], dtype=th.long),

        num_real_nodes=th.tensor([num_real_nodes], dtype=th.long),
        num_hyperedges=th.tensor([num_hyperedges], dtype=th.long),

        # Dummy values
        node_attr=th.zeros(num_real_nodes, 2, dtype=th.float32),
        hyperedge_attr=th.zeros(num_hyperedges, 1, dtype=th.float32),
        incidence_attr=th.zeros(edge_index.size(1), 1, dtype=th.float32),
    )

    data.num_nodes = total_nodes

    return data


def load_generated_topologies(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
        
    return data['ema_0.99']['pred_hypergraphs']

# ---------------- DATASET ----------------

class HypergraphPositionDataset(Dataset):
    def __init__(self, hypergraphs):
        self.hypergraphs = [hypergraph_to_data(H, extract_node_features=True) for H in hypergraphs]

    def __len__(self):
        return len(self.hypergraphs)

    def __getitem__(self, idx):
        return self.hypergraphs[idx]

# ---------------- TRAIN / SAMPLE ----------------

def train_position_model(model, sign_net, train_dataset):
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
        collate_fn=Batch.from_data_list,
    )

    model = model.to(DEVICE)
    sign_net = sign_net.to(DEVICE)

    opt = optim.AdamW(model.parameters(), lr=LR)
    mse = nn.MSELoss()

    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for batch_data in train_loader:
            batch_data = batch_data.to(DEVICE)

            x1 = batch_data.node_features
            x0 = th.randn_like(x1)

            t = th.rand((), device=DEVICE)
            x_t = (1.0 - t) * x0 + t * x1

            target_v = x1 - x0

            spectral_features = batch_data.spectral_features
            edge_index = batch_data.edge_index
        
            all_emb = sign_net(
                spectral_features=spectral_features,
                edge_index=edge_index,
            )
        
            node_emb = all_emb[batch_data.node_type == 1]
            hyperedge_emb = all_emb[batch_data.node_type == 0]
        
            noise_cond = th.full(
                (int(batch_data.num_graphs),),
                float(t),
                device=x_t.device,
                dtype=th.float32,
            )
        
            node_features_initial = th.zeros_like(x_t)
        
            _, _, _, pred_v, _ = model(
                incidence_index=batch_data.edge_index,
                batch=batch_data.batch,
                node_type=batch_data.node_type,
                budget=batch_data.budget,
                node_attr=batch_data.node_attr,
                hyperedge_attr=batch_data.hyperedge_attr,
                incidence_attr=batch_data.incidence_attr,
                node_features_initial=node_features_initial,
                node_features=x_t,
                hyperedge_features_initial=None,
                hyperedge_features=None,
                node_emb=node_emb,
                hyperedge_emb=hyperedge_emb,
                noise_cond=noise_cond,
                red_frac=batch_data.red_frac.view(-1),
                target_size=batch_data.target_size,
            )

            loss = mse(pred_v, target_v)

            opt.zero_grad()
            loss.backward()
            th.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()

        if epoch % 5 == 0 or epoch == EPOCHS - 1:
            print(f"Epoch {epoch:03d} | loss={total_loss / len(train_loader):.6f}")

    return model


@th.no_grad()
def sample_positions(model, sign_net, H):
    model.eval()
    sign_net.eval()

    data = hypergraph_to_data(H, extract_node_features=False)
    batch_data = Batch.from_data_list([data]).to(DEVICE)

    n = int(data.num_real_nodes.item())
    x = th.randn(n, POS_DIM, device=DEVICE)

    dt = 1.0 / SAMPLING_STEPS

    for i in range(SAMPLING_STEPS):
        t = th.tensor(i / SAMPLING_STEPS, device=DEVICE)

        all_emb = sign_net(
            spectral_features=batch_data.spectral_features,
            edge_index=batch_data.edge_index,
        )

        node_emb = all_emb[batch_data.node_type == 1]
        hyperedge_emb = all_emb[batch_data.node_type == 0]

        noise_cond = th.full(
            (int(batch_data.num_graphs),),
            float(t),
            device=x.device,
            dtype=th.float32,
        )

        node_features_initial = th.zeros_like(x)

        _, _, _, v, _ = model(
            incidence_index=batch_data.edge_index,
            batch=batch_data.batch,
            node_type=batch_data.node_type,
            budget=batch_data.budget,
            node_attr=batch_data.node_attr,
            hyperedge_attr=batch_data.hyperedge_attr,
            incidence_attr=batch_data.incidence_attr,
            node_features_initial=node_features_initial,
            node_features=x,
            hyperedge_features_initial=None,
            hyperedge_features=None,
            node_emb=node_emb,
            hyperedge_emb=hyperedge_emb,
            noise_cond=noise_cond,
            red_frac=batch_data.red_frac.view(-1),
            target_size=batch_data.target_size,
        )

        x = x + dt * v

    node_positions = x.detach().cpu().numpy()

    # Reuse the existing generated topology H.
    for i, node in enumerate(H.nodes):
        H.nodes[node].feature = node_positions[i]
        
    return H

# ---------------- LOAD DATA ----------------

with open(DATASET_PATH, "rb") as f:
    data = pickle.load(f)

train_graphs = data["train"]
test_graphs = data["test"]

train_dataset = HypergraphPositionDataset(train_graphs)

# ---------------- METRICS ----------------

metric_objs = [metrics.NodeNumDiff(),
   metrics.NodeDegreeDistrWasserstein(),
   metrics.EdgeSizeDistrWasserstein(),
   metrics.Spectral(),
   metrics.CentralityCloseness(),
   metrics.CentralityBetweenness(),
   metrics.CentralityHarmonic(),
   metrics_mesh.ChamferNearestNeighborDistance(),
   metrics_mesh.ChamferCoverage(),
   metrics_mesh.ChamferDiversity()]

# ---------------- EVALUATE EACH GENERATED TOPOLOGY PKL AS ONE RUN ----------------

all_metrics = []
per_pkl_metrics = {}
all_generated = {}

for run_idx, topology_path in enumerate(GENERATED_TOPOLOGIES_PATHS):
    print(f"\n================ RUN {run_idx}: {topology_path} ================")

    # ---------------- MODELS ----------------
    sign_net = gg.model.SignNet(
        num_eigenvectors=NUM_EIGENVECTORS,
        hidden_dim=SIGN_NET_CFG["hidden_dim"],
        out_dim=MODEL_CFG["expansion_emb_dim"],
        num_layers=SIGN_NET_CFG["num_layers"],
        dropout=MODEL_CFG["dropout"],
    )
    
    model = gg.model.SparsePPGN(
        node_features_in_dim=POS_DIM,
        node_features_out_dim=POS_DIM,
        node_features_emb_dim=MODEL_CFG["node_features_emb_dim"],
    
        # no hyperedge features
        hyperedge_features_in_dim=0,
        hyperedge_features_out_dim=0,
        hyperedge_features_emb_dim=0,
    
        expansion_node_in_dim=2,
        expansion_hyperedge_in_dim=1,
        expansion_incidence_in_dim=1,
    
        # fixed topology: do not predict expansion / incidence / edge outputs
        expansion_node_out_dim=0,
        expansion_hyperedge_out_dim=0,
        expansion_incidence_out_dim=0,
    
        expansion_emb_dim=MODEL_CFG["expansion_emb_dim"],
        hidden_dim=MODEL_CFG["hidden_dim"],
        ppgn_dim=MODEL_CFG["ppgn_dim"],
        num_layers=MODEL_CFG["num_layers"],
        dropout=MODEL_CFG["dropout"],
        self_conditioning=False,
    )

    # ---------------- TRAIN ----------------
    print("\n================ TRAINING ================")
    position_model = train_position_model(model, sign_net, train_dataset)

    print("\n================ GENERATING FEATURES ================")
    generated_topologies = load_generated_topologies(topology_path)

    gen_graphs = [
        sample_positions(model, sign_net, H)
        for H in generated_topologies
    ]

    run_metrics = {}

    for m in metric_objs:
        val = m(test_graphs, gen_graphs, train_graphs)
        run_metrics[str(m)] = float(val)
        print(f"{m}: {val:.6f}")

    name = os.path.splitext(os.path.basename(topology_path))[0]
    out_subdir = os.path.join(OUTPUT_DIR, f"run_{run_idx}_{name}")
    os.makedirs(out_subdir, exist_ok=True)

    with open(os.path.join(out_subdir, "generated_topologies_with_positions.pkl"), "wb") as f:
        pickle.dump(gen_graphs, f)

    with open(os.path.join(out_subdir, "metrics.txt"), "w") as f:
        f.write(f"Dataset: {DATASET_PATH}\n")
        f.write(f"Generated topologies: {topology_path}\n")
        f.write("Position model trained once. This PKL is treated as one run.\n\n")

        for key, val in run_metrics.items():
            f.write(f"{key}: {val:.6f}\n")

    all_metrics.append(run_metrics)
    per_pkl_metrics[topology_path] = run_metrics
    all_generated[topology_path] = gen_graphs

# ---------------- SUMMARY MEAN / STD ACROSS PKLS ----------------

summary_path = os.path.join(OUTPUT_DIR, "metrics_summary_across_topology_pkls.txt")

with open(os.path.join(OUTPUT_DIR, "all_generated_with_positions_by_pkl.pkl"), "wb") as f:
    pickle.dump(all_generated, f)

with open(os.path.join(OUTPUT_DIR, "per_pkl_metrics.pkl"), "wb") as f:
    pickle.dump(per_pkl_metrics, f)

print("\n================ SUMMARY ACROSS TOPOLOGY PKLS ================")

with open(summary_path, "w") as f:
    f.write(f"Dataset: {DATASET_PATH}\n")
    f.write("Position model trained once.\n")
    f.write("Each generated-topology PKL is treated as one run.\n\n")

    f.write("Input PKLs:\n")
    for p in GENERATED_TOPOLOGIES_PATHS:
        f.write(f"- {p}\n")
    f.write("\n")

    keys = all_metrics[0].keys()

    for key in keys:
        vals = np.array([run[key] for run in all_metrics], dtype=np.float64)
        mean = vals.mean()
        std = vals.std()

        line = f"{key}: mean={mean:.6f}, std={std:.6f}"
        print(line)
        f.write(line + "\n")

    f.write("\nPer-PKL metrics:\n")
    for topology_path, result in per_pkl_metrics.items():
        f.write(f"\n=== {topology_path} ===\n")
        for key, val in result.items():
            f.write(f"{key}: {val:.6f}\n")

print(f"\nSaved all results to: {OUTPUT_DIR}")